In [ ]:
import sys
sys.path.insert(0, '..')
from dependencies import *

models = get_models(exclude=['envelope_log_8band'])

In [ ]:
# ESTIMATE TRF FOR DECODER

subject_model_predictors = eelbrain.load.unpickle(PREDICTOR_DIR / f'~concatenated_predictors.pickle')

for subject in SUBJECTS:
    # ————————————————————————————————————————————————————————————————————————————————————————————————
    # Make save directory
    subject_trf_dir = TRF_DIR / subject
    subject_trf_dir.mkdir(exist_ok=True)
    trf_paths = {model: subject_trf_dir / f'{subject} decoder-{model}.pickle' for model in models}

    # ————————————————————————————————————————————————————————————————————————————————————————————————
    # Load eeg
    eeg_concatenated = eelbrain.load.unpickle(EEG_DIR / subject / f'{subject}concatenated_eeg.pickle')
    print('——————————————————————————————')
    print(f'{subject} eeg extracted')
    
    for model, predictors in models.items():
        path = trf_paths[model]
        
        # ————————————————————————————————————————————————————————————————————————————————————————————————
        # Skip if the file already exists
        #'''
        skip = False
        if path.exists():
            try:
                existing_trf = eelbrain.load.unpickle(path)
                if existing_trf is not None:
                    print(f"TRF for {subject} ~ {model} already exists, skipping...\n")
                    skip = True
            except Exception as e:
                print(f"Warning: existing TRF file for {subject} ~ {model} is corrupted or unreadable, will recompute. ({e})")
        if skip:
            continue
        print(f"Estimating: {subject} ~ {model}")
        #'''
        # ————————————————————————————————————————————————————————————————————————————————————————————————
        # Fit the mTRF
        predictors_concatenated = subject_model_predictors[subject][model][0] # <-- index 0 to match dimensions
        print(eeg_concatenated, 'eeg concatenated')
        print(predictors_concatenated, 'predictors_concatenated')
        print('now boosting')
        trf_decoder = eelbrain.boosting(predictors_concatenated, eeg_concatenated, -1.000, 0.100, 
                                error='l1', basis=0.050, partitions=5, test=1, selective_stopping=True)

        # ————————————————————————————————————————————————————————————————————————————————————————————————
        # Save in directory
        eelbrain.save.pickle(trf_decoder, path)
        print(f'TRF for {model} complete')

——————————————————————————————
S05 eeg extracted
TRF for S05 ~ envelope_log already exists, skipping...

TRF for S05 ~ envelope_onset already exists, skipping...

TRF for S05 ~ envelope_log_onset already exists, skipping...

——————————————————————————————
S34 eeg extracted
TRF for S34 ~ envelope_log already exists, skipping...

TRF for S34 ~ envelope_onset already exists, skipping...

TRF for S34 ~ envelope_log_onset already exists, skipping...

——————————————————————————————
S35 eeg extracted
TRF for S35 ~ envelope_log already exists, skipping...

TRF for S35 ~ envelope_onset already exists, skipping...

TRF for S35 ~ envelope_log_onset already exists, skipping...

——————————————————————————————
S03 eeg extracted
TRF for S03 ~ envelope_log already exists, skipping...

TRF for S03 ~ envelope_onset already exists, skipping...

TRF for S03 ~ envelope_log_onset already exists, skipping...

——————————————————————————————
S04 eeg extracted
TRF for S04 ~ envelope_log already exists, skipping